In [ ]:
import re, itertools
import pandas as pd
import tqdm
from googletrans import Translator

tqdm.pandas()
def detect_encoding(path):
    for enc in ("utf-8", "gbk", "cp936", "latin1"):
        try:
            with open(path, encoding=enc, errors="strict") as f:
                f.read(1024)
            return enc
        except Exception:
            continue
    return "utf-8"

def detect_delim(path, enc):
    with open(path, encoding=enc, errors="replace") as f:
        sample = "".join(itertools.islice(f, 20))
    counts = {'\t': sample.count('\t'), ',': sample.count(','), ';': sample.count(';')}
    return max(counts, key=counts.get)

def extract_cas(s):
    if pd.isna(s): return ""
    s = str(s)
    m = re.search(r'\d{1,7}-\d{1,2}-\d', s)
    return m.group(0) if m else s.strip()

def main():
    inp = r"e:\理化所文献\微调\pva_post_process\cycle\materials\materials.csv"
    out = r"e:\理化所文献\微调\pva_post_process\cycle\materials\materials_en.csv"

    enc = detect_encoding(inp)
    sep = detect_delim(inp, enc)

    df = pd.read_csv(inp, sep=sep, header=None, names=["name", "cas"], encoding=enc, engine="python", dtype=str)
    df["cas"] = df["cas"].apply(extract_cas)

    translator = Translator()
    def to_en(text):
        if not text or str(text).strip()=="":
            return ""
        try:
            det = translator.detect(text)
            if det.lang == "en":
                return str(text).strip()
            return translator.translate(text, dest="en").text
        except Exception:
            return str(text).strip()

    df["name_en"] = df["name"].fillna("").progress_apply(to_en)
    df_out = df[["name_en", "cas"]].rename(columns={"name_en": "Name_en", "cas": "CAS"})
    df_out.to_csv(out, index=False, encoding="utf-8-sig")
    print("saved ->", out)

if __name__ == "__main__":
    main()

saved -> e:\理化所文献\微调\pva_post_process\cycle\materials\materials_en.csv
